In [1]:
import os
os.chdir('/home/elious/research_projects/mdpi_sensors_2026')

# Notebook 12: Per-Category Recall (CICIoT2023)

For each attack category: count TP and FN, compute recall.  
Categories: DDoS, DoS, Reconnaissance, Brute Force, Spoofing, Mirai, Web Attacks.

The `label` column in cic_test.csv contains the original multi-class labels.

In [2]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [3]:
cic_test = pd.read_csv('data/processed/cic_test.csv')

X_te = cic_test.drop(columns=['label', 'label_binary'])
y_te_binary = cic_test['label_binary'].values
labels_orig = cic_test['label'].values  # multi-class labels

print('Test shape:', X_te.shape)
print('Unique categories:', pd.Series(labels_orig).nunique())
print(pd.Series(labels_orig).value_counts())

Test shape: (39995, 46)
Unique categories: 32
DDoS-ICMP_Flood            6163
DDoS-UDP_Flood             4705
DDoS-TCP_Flood             3891
DDoS-PSHACK_Flood          3545
DDoS-SYN_Flood             3502
DDoS-RSTFINFlood           3486
DDoS-SynonymousIP_Flood    3092
DoS-UDP_Flood              2757
DoS-TCP_Flood              2266
DoS-SYN_Flood              1724
BenignTraffic               946
Mirai-greeth_flood          834
Mirai-udpplain              720
Mirai-greip_flood           663
DDoS-ICMP_Fragmentation     345
MITM-ArpSpoofing            266
DDoS-UDP_Fragmentation      266
DDoS-ACK_Fragmentation      254
DNS_Spoofing                153
Recon-OSScan                 99
Recon-HostDiscovery          90
Recon-PortScan               77
DoS-HTTP_Flood               48
VulnerabilityScan            28
DDoS-SlowLoris               23
DDoS-HTTP_Flood              23
DictionaryBruteForce         11
BrowserHijacking              6
SqlInjection                  4
CommandInjection          

In [4]:
# Map fine-grained labels to 7 high-level categories per EXPERIMENT_CONFIG.md
def map_category(label):
    label = str(label)
    if label == 'BenignTraffic':
        return 'Benign'
    if 'DDoS' in label:
        return 'DDoS'
    if 'DoS' in label:
        return 'DoS'
    if 'Recon' in label or 'VulnerabilityScan' in label:
        return 'Reconnaissance'
    if 'BruteForce' in label or 'DictionaryBrute' in label:
        return 'Brute Force'
    if 'Spoof' in label or 'MITM' in label or 'DNS_Spoofing' in label:
        return 'Spoofing'
    if 'Mirai' in label:
        return 'Mirai'
    if 'Injection' in label or 'XSS' in label or 'Hijacking' in label or 'Sql' in label:
        return 'Web Attacks'
    return 'Other'

categories = pd.Series(labels_orig).map(map_category)
print('\nCategory distribution:')
print(categories.value_counts())


Category distribution:
DDoS              29295
DoS                6795
Mirai              2217
Benign              946
Spoofing            419
Reconnaissance      295
Web Attacks          17
Brute Force          11
Name: count, dtype: int64


In [5]:
MODELS = ['RandomForest', 'DecisionTree', 'XGBoost', 'LogisticRegression']
all_rows = []

for model in MODELS:
    pred_df = pd.read_csv(f'results/baselines/cic_{model}_predictions.csv')
    y_pred = pred_df['y_pred'].values

    for cat in ['DDoS', 'DoS', 'Reconnaissance', 'Brute Force', 'Spoofing', 'Mirai', 'Web Attacks']:
        mask = (categories == cat).values
        if mask.sum() == 0:
            continue

        y_true_cat = y_te_binary[mask]
        y_pred_cat = y_pred[mask]

        # All attack samples: label_binary = 1
        tp = int(np.sum((y_true_cat == 1) & (y_pred_cat == 1)))
        fn = int(np.sum((y_true_cat == 1) & (y_pred_cat == 0)))
        total = int(mask.sum())
        recall = round(tp / (tp + fn), 6) if (tp + fn) > 0 else 0

        all_rows.append({
            'model': model,
            'category': cat,
            'total_samples': total,
            'tp': tp,
            'fn': fn,
            'recall': recall
        })

df_cat = pd.DataFrame(all_rows)
df_cat.to_csv('results/per_category_recall_cic.csv', index=False)
print('Saved results/per_category_recall_cic.csv')

# Pivot for readability
pivot = df_cat.pivot(index='category', columns='model', values='recall')
print('\nPer-category recall:')
print(pivot.round(4))

Saved results/per_category_recall_cic.csv

Per-category recall:
model           DecisionTree  LogisticRegression  RandomForest  XGBoost
category                                                               
Brute Force           0.8182              0.2727        0.9091   0.4545
DDoS                  1.0000              0.9945        1.0000   1.0000
DoS                   1.0000              0.9894        1.0000   1.0000
Mirai                 1.0000              0.9765        1.0000   1.0000
Reconnaissance        0.9017              0.3797        0.9085   0.7186
Spoofing              0.9069              0.3389        0.9165   0.7232
Web Attacks           0.7059              0.1765        0.9412   0.5294


In [6]:
print('Notebook 12 complete.')

Notebook 12 complete.
